In [2]:
import psycopg2
from dotenv import load_dotenv
import os
import pandas as pd
import re

# Load environment variables from .env
load_dotenv()

# Fetch variables
DBUSER = os.getenv("dbuser")
PASSWORD = os.getenv("password")
HOST = os.getenv("host")
PORT = os.getenv("port")
DBNAME = os.getenv("dbname")


# Connect to the database with timeout
def run_query_and_save_csv(query, columns, output_path="output/products.csv"):
    try:
        print(f"Attempting to connect to {HOST}:{PORT}...")
        connection = psycopg2.connect(
            user=DBUSER,
            password=PASSWORD,
            host=HOST,
            port=PORT,
            dbname=DBNAME,
            connect_timeout=10  # 10 second timeout
        )
        print("Connection successful!")

        cursor = connection.cursor()
        cursor.execute(query)
        result = cursor.fetchall()
        df = pd.DataFrame(result, columns=columns)
        df.to_csv(output_path, index=False)
        print(f"Results written to {output_path}")

        cursor.close()
        connection.close()
        print("Connection closed.")

    except Exception as e:
        print(f"Failed to connect: {e}")

python-dotenv could not parse statement starting at line 6


In [ ]:
# First, Categories
with open("sql/product_categories.sql", "r") as file:
    query = file.read()
columns = [
    "name",
]
run_query_and_save_csv(query, columns=columns, output_path="output/product_categories.csv")

Attempting to connect to aws-0-ap-southeast-2.pooler.supabase.com:5432...
Connection successful!
Results written to output/product_categories.csv
Connection closed.


In [8]:
# Pricelist
with open("sql/pricelist.sql", "r") as file:
    query = file.read()
columns = [
    "name",
    "currency_id",
    "item_ids/compute_price",
    "item_ids/base",
    "item_ids/applied_on",
    "item_ids/categ_id",
    "item_ids/price_markup",
    "item_ids/price_surcharge"
]
run_query_and_save_csv(query, columns=columns, output_path="output/nz_pricelist.csv")

Attempting to connect to aws-0-ap-southeast-2.pooler.supabase.com:5432...
Connection successful!
Results written to output/nz_pricelist.csv
Connection closed.
Connection successful!
Results written to output/nz_pricelist.csv
Connection closed.


In [3]:
# Images
## union product images and category images
with open("sql/images.sql", "r") as file:
    query = file.read()
columns = [
    "code",
    "filename",
    "priority"
]
run_query_and_save_csv(query, columns=columns, output_path="output/images.csv")
## store all images locally?

Attempting to connect to aws-0-ap-southeast-2.pooler.supabase.com:5432...
Connection successful!
Results written to output/images.csv
Connection closed.


In [18]:
# Products
base_url = "https://zbbubgexytgtmoamdott.supabase.co/storage/v1/object/public/images/"
supabase = pd.read_csv("output/images.csv")
outputpath = "output/misc_products/"
os.makedirs(outputpath, exist_ok=True)
# Need to ecommerce categories, websites (; delimited)
with open("sql/products.sql", "r") as file:
    query = file.read()
columns = [
    "name",
    "internal reference",
    "Unit",
    "weight",
    "product category",
    "Cost",
    "is_published",
    "ecommerce_category_id",
    "websites"
]

try:
    print(f"Attempting to connect to {HOST}:{PORT}...")
    connection = psycopg2.connect(
        user=DBUSER,
        password=PASSWORD,
        host=HOST,
        port=PORT,
        dbname=DBNAME,
        connect_timeout=10  # 10 second timeout
    )
    print("Connection successful!")

    cursor = connection.cursor()
    cursor.execute(query)
    result = cursor.fetchall()
    df = pd.DataFrame(result, columns=columns)

    cursor.close()
    connection.close()
    print("Connection closed.")

except Exception as e:
    print(f"Failed to connect: {e}")

print(f"Loaded {len(df)} products from database")
# Now add images and categories
for col in ["Image", "Extra Product Media/Name", "Extra Product Media/Image", "Extra Product Media/Sequence"]:
    if col not in df.columns:
        df[col] = ""

# Split into multiple files - track current file and rows
current_file_rows = []
file_counter = 1
rows_per_file = 500
total_rows_exported = 0

# Iterate over each product in odoo
for idx, product in df.iterrows():
    # FIX: Use the correct column name "internal reference" instead of "default_code"
    code = product.get("internal reference")
    # Find all matching images in supabase
    found = supabase[supabase["code"] == code].copy()
    
    # Start tracking rows for this product
    product_rows = []
    
    if found.empty:
        # Still add the product even if no images found
        product_rows.append(product)
    else:
        # Sort by priority (lower number = higher priority)
        found = found.sort_values(by="priority")
        # Assign images according to new logic
        filenames = found["filename"].tolist()
        # Remove empty strings
        filenames = [f for f in filenames if f != ""]
        # Regex to match filenames with no extension (no dot)
        no_ext_regex = re.compile(r'^[^.]+$')
        # Example usage: filter filenames with no extension
        filenames = [f for f in filenames if not no_ext_regex.match(f)]
        priorities = found["priority"].tolist()
        # First image always goes to 'Image'
        product["Image"] = base_url + filenames[0] if len(filenames) > 0 else ""
        # Second image (if present) goes to 'Extra Product Media/Image' in the same row
        if len(filenames) > 1:
            product["Extra Product Media/Image"] = (base_url + filenames[1]).strip()
            product["Extra Product Media/Name"] = filenames[1]
            product["Extra Product Media/Sequence"] = priorities[1]
        else:
            product["Extra Product Media/Image"] = ""
            product["Extra Product Media/Name"] = ""
            product["Extra Product Media/Sequence"] = ""
        product_rows.append(product)
        # If more images, add new rows for each extra image, inserted directly below
        for i in range(2, len(filenames)):
            blank_row = pd.Series({col: "" for col in df.columns})
            blank_row["Extra Product Media/Name"] = filenames[i]
            blank_row["Extra Product Media/Image"] = (base_url + filenames[i]).strip()
            blank_row["Extra Product Media/Sequence"] = priorities[i]
            product_rows.append(blank_row)
    
    # Check if adding this product would exceed the file size limit
    if len(current_file_rows) + len(product_rows) > rows_per_file and len(current_file_rows) > 0:
        # Save current file
        df_out = pd.DataFrame(current_file_rows, columns=df.columns)
        filename = f"{outputpath}/{file_counter}.csv"
        df_out.to_csv(filename, index=False)
        print(f"Exported {len(df_out)} rows to {filename}")
        total_rows_exported += len(df_out)
        
        # Start new file
        file_counter += 1
        current_file_rows = []
    
    # Add all rows for this product to current file
    current_file_rows.extend(product_rows)

# Save the last file if there are remaining rows
if current_file_rows:
    df_out = pd.DataFrame(current_file_rows, columns=df.columns)
    filename = f"{outputpath}/{file_counter}.csv"
    df_out.to_csv(filename, index=False)
    print(f"Exported {len(df_out)} rows to {filename}")
    total_rows_exported += len(df_out)

print(f"Total exported: {total_rows_exported} rows across {file_counter} files")

Attempting to connect to aws-0-ap-southeast-2.pooler.supabase.com:5432...
Connection successful!
Connection closed.
Loaded 88 products from database
Exported 88 rows to output/misc_products//1.csv
Total exported: 88 rows across 1 files


In [17]:
# BoMs
base_url = "https://zbbubgexytgtmoamdott.supabase.co/storage/v1/object/public/images/"
supabase = pd.read_csv("output/images.csv")
# Need to ecommerce categories, websites (; delimited)
with open("sql/kitset_components.sql", "r") as file:
    query = file.read()
columns = [
    "name",
    "internal reference",
    "BoM Type",
    "BoM Lines/Component",
    "BoM Lines/Quantity",
    "BoM Lines/Unit"
]
run_query_and_save_csv(query, columns=columns, output_path="output/kitset_components.csv")

Attempting to connect to aws-0-ap-southeast-2.pooler.supabase.com:5432...
Connection successful!
Results written to output/kitset_components.csv
Connection closed.


In [ ]:
# Shipments


In [ ]:
# Stock levels

In [7]:
# add dwg no to purchase description field

# from supabase - name, code, dwg no
# Pricelist
with open("sql/drawing_numbers.sql", "r") as file:
    query = file.read()
columns = [
    "name",
    "code",
    "drawing number"
]
run_query_and_save_csv(query, columns=columns, output_path="output/drawing_numbers.csv")
supabase = pd.read_csv("output/drawing_numbers.csv")
# from odoo - productcsv with name, code, purchase description
odoo = pd.read_csv("odoo_drawing_numbers.csv")
# for each code in supabase, find matching code in odoo productcsv, update purchase description to include dwg no
for index, row in supabase.iterrows():
    code = row['code']
    dwg_no = row['drawing number']
    # find matching code in odoo
    match = odoo[odoo['default_code'] == code]
    if not match.empty:
        odoo_index = match.index[0]
        current_desc = odoo.at[odoo_index, 'description_purchase']
        # append dwg no to purchase description
        
        odoo.at[odoo_index, 'description_purchase'] = f'{dwg_no}'
# save updated odoo csv
odoo.to_csv("output/odoo_products_with_dwg_numbers.csv", index=False)

Attempting to connect to aws-0-ap-southeast-2.pooler.supabase.com:5432...
Connection successful!
Results written to output/drawing_numbers.csv
Connection closed.


In [ ]:
# double spaces in name fields
odoo = pd.read_csv("odoo_drawing_numbers.csv")
# count how many names have double spaces
double_space_count = odoo['name'].str.contains('  ').sum()
print(f"Found {double_space_count} names with double spaces")

# identify rows with double spaces before modification
rows_with_double_spaces = odoo['name'].str.contains('  ')

for index, row in odoo.iterrows():
    name = row['name']
    # replace double spaces with single spaces
    new_name = re.sub(' +', ' ', name)
    odoo.at[index, 'name'] = new_name

# drop the description_purchase column
odoo = odoo.drop('description_purchase', axis=1)

# save only the modified rows (after modification)
odoo[rows_with_double_spaces].to_csv("output/odoo_products_no_double_spaces.csv", index=False)

Found 101 names with double spaces


In [ ]:
# update names
new_names = pd.read_csv("new_names.csv")
odoo_names = pd.read_csv("odoo_names.csv")
# for each code in new_names, find matching code in odoo_names, update name field
output = []
for index, row in new_names.iterrows():
    code = row['internal reference']
    new_name = row['name']
    # find matching code in odoo_names
    match = odoo_names[odoo_names['default_code'] == code]
    if not match.empty:
        if new_name != match.iloc[0]['name']:
            output.append((match.iloc[0]['id'], code, match.iloc[0]['name'], new_name))
        odoo_index = match.index[0]
        odoo_names.at[odoo_index, 'name'] = new_name
output_df = pd.DataFrame(output, columns=['id', 'internal reference', 'old name', 'new name'])
output_df.to_csv("output/odoo_name_changes.csv", index=False)

Updating code CorCnrJoiS-50-45Acu1047 name from '45deg 50mm Acute corner joiner 1050' to '45deg 50mm Acute corner joiner 1050	'
Updating code CorCnrJoiS-50-45Acu597 name from '45deg 50mm Acute corner joiner 600' to '45deg 50mm Acute corner joiner 600	'
Updating code CorCnrJoiS-50-45Ext24 name from '45deg 50mm square corner joiner ext-607' to '45deg 50mm square corner joiner ext-607
'
Updating code CorCnrJoiS-50-45Ext30 name from '45deg 50mm square corner joiner ext-759' to '45deg 50mm square corner joiner ext-759
'
Updating code CorCnrJoiR-50-90Ext12 name from '90deg 50mm round corner joiner ext-302' to '90deg 50mm round corner joiner ext-302
'
Updating code CorPlaW1800-400-400T name from 'Corten Planter 1800L x 400W X 400Tall' to 'Corten Kitset  Planter 1800L x 400W X 400Tall'
Updating code CorPlaW1800-400-600T name from 'Corten Planter 1800L x 400W X 600Tall' to 'Corten Kitset  Planter 1800L x 400W X 600Tall'
Updating code CorPlaW72-16-16T name from 'Corten Planter 1828L x 408W x 408

[('__export__.product_template_3534_721514e1',
  'CorCnrJoiS-50-45Acu1047',
  '45deg 50mm Acute corner joiner 1050',
  '45deg 50mm Acute corner joiner 1050\t'),
 ('__export__.product_template_3564_12ad07e7',
  'CorCnrJoiS-50-45Acu597',
  '45deg 50mm Acute corner joiner 600',
  '45deg 50mm Acute corner joiner 600\t'),
 ('__export__.product_template_2959_88d0d484',
  'CorCnrJoiS-50-45Ext24',
  '45deg 50mm square corner joiner ext-607',
  '45deg 50mm square corner joiner ext-607\r\n'),
 ('__export__.product_template_2958_44e8dd18',
  'CorCnrJoiS-50-45Ext30',
  '45deg 50mm square corner joiner ext-759',
  '45deg 50mm square corner joiner ext-759\r\n'),
 ('__export__.product_template_2932_158ff6f1',
  'CorCnrJoiR-50-90Ext12',
  '90deg 50mm round corner joiner ext-302',
  '90deg 50mm round corner joiner ext-302\r\n'),
 ('__export__.product_template_4553_d4f79d6c',
  'CorPlaW1800-400-400T',
  'Corten Planter 1800L x 400W X 400Tall',
  'Corten Kitset  Planter 1800L x 400W X 400Tall'),
 ('__exp